# Modeling K-Means Clustering
## Sistem Clustering Kemiskinan Wilayah di Indonesia

Notebook ini berisi tahap **modeling** dari pipeline proyek, melanjutkan dari hasil data cleaning.

Alur modeling:
1. Load dataset
2. Data Cleaning (missing value, outlier, duplikat)
3. Seleksi fitur & Min-Max Scaling (manual)
4. Elbow Method — cari K optimal
5. Silhouette Score — konfirmasi K optimal
6. Training K-Means Manual
7. Interpretasi & pelabelan cluster
8. Visualisasi hasil
9. Simpan output untuk tim Geospatial

---
## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import warnings
warnings.filterwarnings('ignore')

---
## 2. Load Dataset

In [ ]:
df = pd.read_excel('/content/Data_Tingkat_Kemiskinan.xlsx')

print('Shape data:', df.shape)
print()
df.head()

In [ ]:
df.info()

---
## 3. Data Cleaning

Meskipun sudah ada tim cleaning, tahap ini tetap dijalankan sebagai validasi agar modeling tidak error.

In [ ]:
# --- Missing Value ---
print('Missing value per kolom:')
print(df.isnull().sum())

df = df.dropna()
print('\nSetelah dropna, shape:', df.shape)

In [ ]:
# --- Duplikat ---
print('Jumlah duplikat:', df.duplicated().sum())
df = df.drop_duplicates()
print('Shape setelah drop duplikat:', df.shape)

In [ ]:
# --- Outlier: IQR method, drop pada fitur yang akan dipakai ---
fitur_outlier = [
    'Tingkat_Penduduk_Miskin',
    'PengeluaranPerKapita',
    'Usia Harapan Hidup'
]

def hapus_outlier_iqr(data, kolom):
    df_bersih = data.copy()
    for col in kolom:
        Q1 = df_bersih[col].quantile(0.25)
        Q3 = df_bersih[col].quantile(0.75)
        IQR = Q3 - Q1
        batas_bawah = Q1 - 1.5 * IQR
        batas_atas  = Q3 + 1.5 * IQR
        sebelum = len(df_bersih)
        df_bersih = df_bersih[(df_bersih[col] >= batas_bawah) & (df_bersih[col] <= batas_atas)]
        print(f'  {col}: hapus {sebelum - len(df_bersih)} baris outlier')
    return df_bersih

print('Proses hapus outlier:')
df = hapus_outlier_iqr(df, fitur_outlier)
df = df.reset_index(drop=True)
print(f'\nShape setelah hapus outlier: {df.shape}')

---
## 4. Seleksi Fitur

Tiga fitur dipilih karena paling merepresentasikan kondisi kemiskinan suatu wilayah:

| Fitur | Alasan |
|---|---|
| `Tingkat_Penduduk_Miskin` | Indikator langsung kemiskinan |
| `PengeluaranPerKapita` | Proxy daya beli / kesejahteraan ekonomi |
| `Usia Harapan Hidup` | Proxy kualitas hidup & layanan kesehatan |

In [ ]:
features = [
    'Tingkat_Penduduk_Miskin',
    'PengeluaranPerKapita',
    'Usia Harapan Hidup'
]

X = df[features].copy()
print('Statistik fitur yang digunakan:')
X.describe()

---
## 5. Min-Max Scaling Manual

K-Means menghitung **jarak Euclidean** antar titik data. Kalau skala fitur beda jauh (misalnya persen vs ribuan rupiah), fitur dengan angka besar akan mendominasi perhitungan — hasil cluster jadi bias.

**Analogi:** Bayangkan kamu gabungkan pengukuran berat badan (50–100 kg) dengan tinggi badan (155–180 cm). Selisih berat 50 kg vs selisih tinggi 25 cm terasa sama-sama besar, tapi di mata algoritma, angka 50 jauh lebih besar dari 25. Scaling membuat semua fitur 'berbicara dengan volume yang sama'.

Formula: `X_scaled = (X - X_min) / (X_max - X_min)` → hasil antara 0 dan 1.

In [ ]:
def minmax_scale(X):
    """
    Min-Max Normalization manual.
    Input : numpy array (n_samples, n_features)
    Output: array ternormalisasi [0,1], x_min, x_max tiap fitur
    """
    x_min = X.min(axis=0)
    x_max = X.max(axis=0)
    x_scaled = (X - x_min) / (x_max - x_min)
    return x_scaled, x_min, x_max

X_arr = X.values
X_scaled, x_min, x_max = minmax_scale(X_arr)

print('Min tiap fitur:', x_min)
print('Max tiap fitur:', x_max)
print()

X_scaled_df = pd.DataFrame(X_scaled, columns=features)
print('Tampilan 5 baris setelah scaling:')
X_scaled_df.head()

In [ ]:
X_scaled_df.describe()

---
## 6. Implementasi K-Means Manual

**Cara kerja K-Means (analogi):**

Bayangkan kamu punya ratusan titik di peta dan ingin membaginya jadi K kelompok wilayah:
1. Lempar K bendera (centroid) secara acak ke peta
2. Setiap titik 'bergabung' ke bendera terdekat → terbentuk K kelompok
3. Geser tiap bendera ke titik tengah (rata-rata) kelompoknya
4. Ulangi langkah 2–3 sampai bendera tidak bergerak lagi (konvergen)

Sesederhana itu.

In [ ]:
def euclidean_distance(a, b):
    """Hitung jarak Euclidean antara dua titik."""
    return np.sqrt(np.sum((a - b) ** 2))


def kmeans_manual(X, k, max_iter=300, tol=1e-4, random_state=42):
    """
    Implementasi K-Means Clustering dari nol (tanpa sklearn).

    Parameters
    ----------
    X            : numpy array (n_samples, n_features)
    k            : jumlah cluster
    max_iter     : batas maksimum iterasi
    tol          : toleransi konvergensi — berhenti jika pergeseran centroid < tol
    random_state : seed agar hasil bisa direproduksi

    Returns
    -------
    labels    : array label cluster tiap data point
    centroids : posisi akhir centroid
    inertia   : total within-cluster sum of squares (WCSS)
    """
    np.random.seed(random_state)
    n_samples = X.shape[0]

    # Langkah 1: Inisialisasi — pilih k titik data secara acak sebagai centroid awal
    idx_awal = np.random.choice(n_samples, k, replace=False)
    centroids = X[idx_awal].copy().astype(float)

    labels = np.zeros(n_samples, dtype=int)

    for iterasi in range(max_iter):

        # Langkah 2: Assign setiap titik ke centroid terdekat
        labels_baru = np.zeros(n_samples, dtype=int)
        for i in range(n_samples):
            jarak = [euclidean_distance(X[i], centroids[c]) for c in range(k)]
            labels_baru[i] = np.argmin(jarak)

        # Langkah 3: Hitung centroid baru = rata-rata tiap cluster
        centroids_baru = np.zeros_like(centroids)
        for c in range(k):
            anggota = X[labels_baru == c]
            if len(anggota) == 0:
                # cluster kosong: reinisialisasi ke titik acak
                centroids_baru[c] = X[np.random.randint(n_samples)]
            else:
                centroids_baru[c] = anggota.mean(axis=0)

        # Langkah 4: Cek konvergensi
        pergeseran = max([euclidean_distance(centroids_baru[c], centroids[c]) for c in range(k)])

        labels    = labels_baru
        centroids = centroids_baru

        if pergeseran < tol:
            print(f'  Konvergen pada iterasi ke-{iterasi + 1} (pergeseran = {pergeseran:.6f})')
            break

    # Hitung inertia (WCSS = Within-Cluster Sum of Squares)
    inertia = sum(
        euclidean_distance(X[i], centroids[labels[i]]) ** 2
        for i in range(n_samples)
    )

    return labels, centroids, inertia


print('Fungsi K-Means manual siap.')

---
## 7. Elbow Method — Menentukan K Optimal

**Cara baca:** Kita jalankan K-Means untuk K = 1–10, lalu plot nilai **inertia** (total jarak kuadrat tiap titik ke centroid clusternya). Di titik di mana penurunan inertia tiba-tiba melambat (seperti bentuk siku/elbow), itulah K terbaik — menambah cluster lebih banyak tidak memberi manfaat signifikan.

**Analogi:** Seperti nambah karyawan di kantor. Awalnya tiap karyawan baru sangat membantu, tapi di titik tertentu menambah karyawan lagi tidak terlalu berpengaruh — itulah titik optimalnya.

In [ ]:
inertia_list = []
K_range = range(1, 11)

print('Menjalankan Elbow Method (K=1 sampai 10)...')
for k in K_range:
    print(f'  K={k}', end=' ')
    _, _, inertia = kmeans_manual(X_scaled, k=k, random_state=42)
    inertia_list.append(inertia)
    print(f'→ Inertia: {inertia:.4f}')

print('\nSelesai.')

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(list(K_range), inertia_list, marker='o', color='steelblue', linewidth=2, markersize=8)
plt.xlabel('Jumlah Cluster (K)', fontsize=12)
plt.ylabel('Inertia (WCSS)', fontsize=12)
plt.title('Elbow Method — Menentukan K Optimal', fontsize=14)
plt.xticks(list(K_range))
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

---
## 8. Silhouette Score Manual — Konfirmasi K Optimal

**Apa itu Silhouette Score?**
Ukuran seberapa 'pas' sebuah data berada di clusternya sendiri dibanding cluster tetangga.

- Nilainya antara **-1 sampai 1**
- **Mendekati 1** → titik sudah di cluster yang tepat
- **Mendekati 0** → titik ada di perbatasan dua cluster
- **Negatif** → titik kemungkinan salah cluster

Rumus untuk setiap titik `i`:
- `a(i)` = rata-rata jarak ke sesama anggota cluster sendiri
- `b(i)` = rata-rata jarak ke anggota cluster **terdekat lainnya**
- `s(i) = (b - a) / max(a, b)`

**Analogi:** Kamu baru pindah ke komplek baru. `a` = rata-rata kedekatan kamu sama tetangga sekomplek. `b` = rata-rata kedekatan kamu sama orang komplek lain yang paling mirip. Kalau `b >> a`, berarti kamu sudah di komplek yang paling cocok.

In [ ]:
def silhouette_score_manual(X, labels):
    """
    Hitung Silhouette Score secara manual.
    """
    n = X.shape[0]
    cluster_ids = np.unique(labels)
    s_list = []

    for i in range(n):
        cluster_i = labels[i]
        anggota_sendiri = X[labels == cluster_i]

        # a(i): rata-rata jarak ke sesama anggota cluster sendiri
        if len(anggota_sendiri) == 1:
            s_list.append(0)
            continue
        a_i = np.mean([
            euclidean_distance(X[i], anggota_sendiri[j])
            for j in range(len(anggota_sendiri))
            if not np.array_equal(anggota_sendiri[j], X[i])
        ])

        # b(i): rata-rata jarak ke cluster terdekat lainnya
        b_i = np.inf
        for c in cluster_ids:
            if c == cluster_i:
                continue
            anggota_lain = X[labels == c]
            rata_jarak = np.mean([
                euclidean_distance(X[i], anggota_lain[j])
                for j in range(len(anggota_lain))
            ])
            if rata_jarak < b_i:
                b_i = rata_jarak

        s_i = (b_i - a_i) / max(a_i, b_i)
        s_list.append(s_i)

    return np.mean(s_list)


print('Menghitung Silhouette Score untuk K=2 sampai 6...')
print('(Proses ini butuh beberapa menit karena dihitung manual)\n')

sil_scores = {}
for k in range(2, 7):
    labels_tmp, _, _ = kmeans_manual(X_scaled, k=k, random_state=42)
    score = silhouette_score_manual(X_scaled, labels_tmp)
    sil_scores[k] = score
    print(f'K={k}  →  Silhouette Score: {score:.4f}')

best_k = max(sil_scores, key=sil_scores.get)
print(f'\nK terbaik berdasarkan Silhouette Score: K = {best_k} (score = {sil_scores[best_k]:.4f})')

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(list(sil_scores.keys()), list(sil_scores.values()),
         marker='s', color='coral', linewidth=2, markersize=8)
plt.xlabel('Jumlah Cluster (K)', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.title('Silhouette Score tiap K', fontsize=14)
plt.xticks(list(sil_scores.keys()))
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

---
## 9. Training Model Final

Gunakan K optimal yang ditemukan dari Elbow + Silhouette Score.

In [ ]:
K_OPTIMAL = best_k  # hasil dari silhouette score
# Jika ingin override manual, ganti dengan: K_OPTIMAL = 3

print(f'Training K-Means dengan K = {K_OPTIMAL}...')
labels_final, centroids_final, inertia_final = kmeans_manual(
    X_scaled, k=K_OPTIMAL, random_state=42
)

df['Cluster'] = labels_final

print(f'\nInertia final : {inertia_final:.4f}')
print(f'\nDistribusi cluster:')
print(df['Cluster'].value_counts().sort_index())

---
## 10. Interpretasi Cluster

Lihat rata-rata tiap fitur per cluster untuk memahami karakteristiknya, lalu beri label yang bermakna.

In [ ]:
# Rata-rata fitur per cluster
ringkasan = df.groupby('Cluster')[features].mean().round(2)
print('Rata-rata fitur per cluster:')
ringkasan

In [ ]:
# Beri label berdasarkan ranking Tingkat_Penduduk_Miskin
# Semakin tinggi % miskin = semakin miskin labelnya
rank_miskin = ringkasan['Tingkat_Penduduk_Miskin'].rank(ascending=True)

# Sesuaikan dengan jumlah K
pilihan_label = {
    2: ['Tidak Miskin', 'Miskin'],
    3: ['Tidak Miskin', 'Miskin', 'Sangat Miskin'],
    4: ['Tidak Miskin', 'Cukup Miskin', 'Miskin', 'Sangat Miskin'],
    5: ['Tidak Miskin', 'Cukup Miskin', 'Miskin', 'Sangat Miskin', 'Ekstrem Miskin'],
    6: ['Tidak Miskin', 'Cukup Miskin', 'Miskin', 'Cukup Sangat Miskin', 'Sangat Miskin', 'Ekstrem Miskin'],
}
daftar_label = pilihan_label.get(K_OPTIMAL, [f'Cluster {i}' for i in range(K_OPTIMAL)])

label_map = {}
for cluster_id, rank_val in rank_miskin.items():
    idx = int(rank_val) - 1
    label_map[cluster_id] = daftar_label[idx]

df['Kategori'] = df['Cluster'].map(label_map)

print('Mapping label cluster:')
for c, lbl in sorted(label_map.items()):
    print(f'  Cluster {c} → {lbl}')

In [ ]:
# Tampilkan ringkasan lengkap
ringkasan['Kategori'] = ringkasan.index.map(label_map)
ringkasan['Jumlah Wilayah'] = df.groupby('Cluster').size()
ringkasan[['Kategori', 'Jumlah Wilayah'] + features]

---
## 11. Visualisasi Hasil Clustering

In [ ]:
# Scatter plot: Tingkat Miskin vs Pengeluaran
warna = plt.cm.get_cmap('Set1', K_OPTIMAL)

plt.figure(figsize=(10, 6))
for c in range(K_OPTIMAL):
    mask = labels_final == c
    plt.scatter(
        X_scaled[mask, 0], X_scaled[mask, 1],
        label=f'Cluster {c} — {label_map.get(c, "")}',
        alpha=0.7, s=60, color=warna(c)
    )

plt.scatter(
    centroids_final[:, 0], centroids_final[:, 1],
    marker='X', s=250, color='black', zorder=5, label='Centroid'
)

plt.xlabel('Tingkat_Penduduk_Miskin (scaled)', fontsize=11)
plt.ylabel('PengeluaranPerKapita (scaled)', fontsize=11)
plt.title(f'Hasil K-Means Manual (K={K_OPTIMAL})\nTingkat Kemiskinan vs Pengeluaran per Kapita', fontsize=13)
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: Tingkat Miskin vs Usia Harapan Hidup
plt.figure(figsize=(10, 6))
for c in range(K_OPTIMAL):
    mask = labels_final == c
    plt.scatter(
        X_scaled[mask, 0], X_scaled[mask, 2],
        label=f'Cluster {c} — {label_map.get(c, "")}',
        alpha=0.7, s=60, color=warna(c)
    )

plt.scatter(
    centroids_final[:, 0], centroids_final[:, 2],
    marker='X', s=250, color='black', zorder=5, label='Centroid'
)

plt.xlabel('Tingkat_Penduduk_Miskin (scaled)', fontsize=11)
plt.ylabel('Usia Harapan Hidup (scaled)', fontsize=11)
plt.title(f'Hasil K-Means Manual (K={K_OPTIMAL})\nTingkat Kemiskinan vs Usia Harapan Hidup', fontsize=13)
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart distribusi jumlah wilayah per cluster
distribusi = df['Kategori'].value_counts().reindex(
    [label_map[c] for c in sorted(label_map.keys())]
)

plt.figure(figsize=(8, 5))
bars = plt.bar(distribusi.index, distribusi.values,
               color=[warna(c) for c in range(K_OPTIMAL)], edgecolor='black')

for bar, val in zip(bars, distribusi.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(val), ha='center', va='bottom', fontsize=11)

plt.xlabel('Kategori Kemiskinan', fontsize=12)
plt.ylabel('Jumlah Wilayah (Kab/Kota)', fontsize=12)
plt.title('Distribusi Wilayah per Cluster Kemiskinan', fontsize=13)
plt.tight_layout()
plt.show()

---
## 12. Sampel Wilayah per Cluster

In [ ]:
kolom_tampil = ['Kabupaten_Kota', 'Kategori'] + features

for c in sorted(df['Cluster'].unique()):
    label_c = label_map.get(c, f'Cluster {c}')
    subset = df[df['Cluster'] == c][kolom_tampil]
    print(f'\n{'='*60}')
    print(f'  CLUSTER {c} — {label_c}  ({len(subset)} wilayah)')
    print(f'{'='*60}')
    print(subset.head(8).to_string(index=False))

---
## 13. Simpan Output untuk Tim Geospatial

In [ ]:
kolom_output = ['Kabupaten_Kota', 'Cluster', 'Kategori'] + features
df_output = df[kolom_output].copy()

# Simpan sebagai CSV untuk tim Geospatial
df_output.to_csv('hasil_clustering_kemiskinan.csv', index=False)

print('File tersimpan: hasil_clustering_kemiskinan.csv')
print(f'Total wilayah  : {len(df_output)}')
print(f'Jumlah cluster : {K_OPTIMAL}')
print()
print('Preview output:')
df_output.head(10)